In [ ]:
import pandas as pd
from rdkit import Chem
from collections import defaultdict

CSV_PATH = '20260415_LNiH2_library_001_to_download_2026_03_24_descriptors_.csv'


def get_substituent_atoms(mol, start_idx, ring_set):
    """Collect all atoms reachable from start_idx without entering ring_set."""
    visited = set()
    stack = [start_idx]
    while stack:
        curr = stack.pop()
        if curr in visited:
            continue
        visited.add(curr)
        for nbr in mol.GetAtomWithIdx(curr).GetNeighbors():
            if nbr.GetIdx() not in ring_set:
                stack.append(nbr.GetIdx())
    return visited


def get_pyridine_substituents(smiles):
    """
    Returns a list of (position, sub_smiles) tuples for every position on the
    pyridine ring at positions 2-5. Unsubstituted positions are reported with
    sub_smiles='H'. Position 6 (oxazoline bond) is ignored.
    Positions are numbered from N (position 1) going away from the oxazoline.

    Uses any-bond SMARTS to match non-aromatic rings (RDKit does not perceive
    aromaticity for these LNiH2 rings due to NH and exocyclic double bond).

    Returns None if no 6-membered N-containing ring is found or if the
    oxazoline attachment cannot be determined.
    """
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None

    pyr = Chem.MolFromSmarts('[#7;r6]1~[#6;r6]~[#6;r6]~[#6;r6]~[#6;r6]~[#6;r6]~1')
    matches = mol.GetSubstructMatches(pyr)
    if not matches:
        return None

    ring_atoms = matches[0]
    ring_set = set(ring_atoms)
    n_idx = next(i for i in ring_atoms if mol.GetAtomWithIdx(i).GetAtomicNum() == 7)

    n_ring_neighbors = [nbr.GetIdx() for nbr in mol.GetAtomWithIdx(n_idx).GetNeighbors()
                        if nbr.GetIdx() in ring_set]

    # Identify the oxazoline: 5-membered rings containing both O and N.
    # Position 6 is the N-neighbor that bonds directly into the oxazoline.
    # Checking for the oxazoline specifically (rather than any ring) avoids
    # false positives when aryl substituents are present at other positions.
    ring_info = mol.GetRingInfo()
    oxazoline_atoms = set()
    for ring in ring_info.AtomRings():
        if len(ring) == 5:
            ring_atom_objs = [mol.GetAtomWithIdx(i) for i in ring]
            if (any(a.GetAtomicNum() == 7 for a in ring_atom_objs) and
                    any(a.GetAtomicNum() == 8 for a in ring_atom_objs)):
                oxazoline_atoms.update(ring)

    def connects_to_oxazoline(atom_idx):
        for nbr in mol.GetAtomWithIdx(atom_idx).GetNeighbors():
            if nbr.GetIdx() not in ring_set and nbr.GetIdx() in oxazoline_atoms:
                return True
        return False

    c6_candidates = [nb for nb in n_ring_neighbors if connects_to_oxazoline(nb)]
    c2_candidates = [nb for nb in n_ring_neighbors if not connects_to_oxazoline(nb)]
    if len(c6_candidates) != 1 or len(c2_candidates) != 1:
        return None

    # Walk the ring N->C2->C3->C4->C5->C6, assigning absolute positions
    positions = {n_idx: 1, c2_candidates[0]: 2}
    prev_idx, curr_idx = n_idx, c2_candidates[0]
    for pos in range(3, 7):
        nexts = [nbr.GetIdx() for nbr in mol.GetAtomWithIdx(curr_idx).GetNeighbors()
                 if nbr.GetIdx() in ring_set and nbr.GetIdx() != prev_idx]
        if not nexts:
            break
        prev_idx, curr_idx = curr_idx, nexts[0]
        positions[curr_idx] = pos

    results = []
    for atom_idx in ring_set:
        pos = positions.get(atom_idx)
        if pos is None or pos == 1 or pos == 6:
            continue  # skip N and the oxazoline-bonded position
        heavy_nbrs = [nbr.GetIdx() for nbr in mol.GetAtomWithIdx(atom_idx).GetNeighbors()
                      if nbr.GetIdx() not in ring_set and nbr.GetAtomicNum() != 1]
        if heavy_nbrs:
            for nbr_idx in heavy_nbrs:
                sub_atoms = get_substituent_atoms(mol, nbr_idx, ring_set)
                sub_smi = Chem.MolFragmentToSmiles(mol, list(sub_atoms))
                results.append((pos, sub_smi))
        else:
            results.append((pos, 'H'))  # unsubstituted position

    return sorted(results)


def get_oxazoline_substituents(smiles):
    """
    Returns a list of (position, sub_smiles) tuples for the two substitutable
    carbons of the oxazoline ring: pos6 (the carbon adjacent to ring N, i.e.
    oxazoline C4) and pos7 (the carbon adjacent to ring O, i.e. oxazoline C5).
    The imine carbon (oxazoline C2, fused to the pyridine) has no free valence
    for a substituent and is excluded. Unsubstituted positions are reported
    with sub_smiles='H'.

    Returns None if no 5-membered O+N ring is found or if the ring carbons
    cannot be unambiguously assigned.
    """
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None

    ring_info = mol.GetRingInfo()
    oxazoline_atoms = None
    for ring in ring_info.AtomRings():
        if len(ring) == 5:
            ring_atom_objs = [mol.GetAtomWithIdx(i) for i in ring]
            if (any(a.GetAtomicNum() == 7 for a in ring_atom_objs) and
                    any(a.GetAtomicNum() == 8 for a in ring_atom_objs)):
                oxazoline_atoms = set(ring)
                break
    if oxazoline_atoms is None:
        return None

    ring_carbons = [i for i in oxazoline_atoms if mol.GetAtomWithIdx(i).GetAtomicNum() == 6]

    def is_imine_carbon(atom_idx):
        atom = mol.GetAtomWithIdx(atom_idx)
        return any(bond.GetOtherAtomIdx(atom_idx) not in oxazoline_atoms and
                   bond.GetBondTypeAsDouble() == 2.0
                   for bond in atom.GetBonds())

    imine_candidates = [i for i in ring_carbons if is_imine_carbon(i)]
    other_carbons = [i for i in ring_carbons if i not in imine_candidates]
    if len(imine_candidates) != 1 or len(other_carbons) != 2:
        return None

    n_idx = next(i for i in oxazoline_atoms if mol.GetAtomWithIdx(i).GetAtomicNum() == 7)

    def adjacent_to_n(atom_idx):
        return any(nbr.GetIdx() == n_idx for nbr in mol.GetAtomWithIdx(atom_idx).GetNeighbors())

    c4_candidates = [i for i in other_carbons if adjacent_to_n(i)]
    c5_candidates = [i for i in other_carbons if not adjacent_to_n(i)]
    if len(c4_candidates) != 1 or len(c5_candidates) != 1:
        return None

    pos_map = {c4_candidates[0]: 6, c5_candidates[0]: 7}

    results = []
    for atom_idx, pos in pos_map.items():
        heavy_nbrs = [nbr.GetIdx() for nbr in mol.GetAtomWithIdx(atom_idx).GetNeighbors()
                      if nbr.GetIdx() not in oxazoline_atoms and nbr.GetAtomicNum() != 1]
        if heavy_nbrs:
            for nbr_idx in heavy_nbrs:
                sub_atoms = get_substituent_atoms(mol, nbr_idx, oxazoline_atoms)
                sub_smi = Chem.MolFragmentToSmiles(mol, list(sub_atoms))
                results.append((pos, sub_smi))
        else:
            results.append((pos, 'H'))  # unsubstituted position

    return sorted(results)

In [8]:
df = pd.read_csv(CSV_PATH)
print(f'Loaded {len(df)} entries')

df['pyridine_subs'] = df['smiles'].apply(get_pyridine_substituents)

no_match = df['pyridine_subs'].isna().sum()
if no_match:
    print(f'Warning: {no_match} entries had no pyridine ring detected')

Loaded 38 entries


In [ ]:
df['oxazoline_subs'] = df['smiles'].apply(get_oxazoline_substituents)

no_match_ox = df['oxazoline_subs'].isna().sum()
if no_match_ox:
    print(f'Warning: {no_match_ox} entries had no oxazoline ring detected')

In [9]:
# Per-entry substituent listing
for idx, row in df.iterrows():
    subs = row['pyridine_subs']
    if subs is None:
        print(f'[{idx}] no pyridine found')
        continue
    parts = [f"pos{p}={s}" for p, s in subs]
    print(f"[{idx}] {'  |  '.join(parts)}")

[0] pos2=H  |  pos3=FC(F)F  |  pos4=H  |  pos5=H
[1] pos2=H  |  pos3=OCc1ccccc1  |  pos4=H  |  pos5=H
[2] pos2=H  |  pos3=FC(F)F  |  pos4=H  |  pos5=H
[3] pos2=C  |  pos3=H  |  pos4=H  |  pos5=H
[4] pos2=H  |  pos3=FC(F)F  |  pos4=H  |  pos5=H
[5] pos2=H  |  pos3=FC(F)F  |  pos4=H  |  pos5=H
[6] pos2=H  |  pos3=FC(F)F  |  pos4=H  |  pos5=H
[7] pos2=H  |  pos3=OCc1ccccc1  |  pos4=H  |  pos5=H
[8] pos2=H  |  pos3=FC(F)F  |  pos4=H  |  pos5=H
[9] pos2=F  |  pos3=H  |  pos4=H  |  pos5=H
[10] pos2=H  |  pos3=FC(F)F  |  pos4=H  |  pos5=H
[11] pos2=H  |  pos3=C  |  pos4=H  |  pos5=H
[12] pos2=H  |  pos3=H  |  pos4=H  |  pos5=H
[13] pos2=H  |  pos3=OC(F)(F)F  |  pos4=H  |  pos5=H
[14] pos2=H  |  pos3=FC(F)F  |  pos4=H  |  pos5=H
[15] pos2=H  |  pos3=FC(F)F  |  pos4=H  |  pos5=H
[16] pos2=H  |  pos3=FC(F)F  |  pos4=H  |  pos5=H
[17] pos2=C  |  pos3=H  |  pos4=H  |  pos5=H
[18] pos2=C  |  pos3=H  |  pos4=H  |  pos5=H
[19] pos2=H  |  pos3=FC(F)F  |  pos4=H  |  pos5=H
[20] pos2=H  |  pos3=FC(F)F  

In [10]:
# Summary: unique substituents at each position
by_pos = defaultdict(list)
for subs in df['pyridine_subs'].dropna():
    for pos, smi in subs:
        by_pos[pos].append(smi)

print('=== Unique substituents by ring position ===\n')
for pos in sorted(by_pos):
    counts = pd.Series(by_pos[pos]).value_counts()
    print(f'POSITION {pos}: {len(counts)} unique')
    for smi, cnt in counts.items():
        print(f'  {smi:40s}  n={cnt}')
    print()

=== Unique substituents by ring position ===

POSITION 2: 4 unique
  H                                         n=32
  C                                         n=4
  F                                         n=1
  cccc                                      n=1

POSITION 3: 9 unique
  FC(F)F                                    n=18
  H                                         n=8
  OCc1ccccc1                                n=5
  CS                                        n=2
  C                                         n=1
  OC(F)(F)F                                 n=1
  F                                         n=1
  cccc                                      n=1
  C#N                                       n=1

POSITION 4: 2 unique
  H                                         n=37
  CO                                        n=1

POSITION 5: 5 unique
  H                                         n=34
  CCS(=O)=O                                 n=1
  CCS                                       n=1

In [11]:
# Wide DataFrame: one row per entry, one column per ring position
POSITIONS = [2, 3, 4, 5]

rows = []
for idx, row in df.iterrows():
    subs = row['pyridine_subs']
    entry = {
        'entry_idx': idx,
        'smiles': row['smiles'],
        'ligand_class': row.get('ligand_class'),
    }
    if subs is None:
        for pos in POSITIONS:
            entry[f'pos{pos}'] = None
    else:
        by_pos = defaultdict(list)
        for pos, smi in subs:
            by_pos[pos].append(smi)
        for pos in POSITIONS:
            entry[f'pos{pos}'] = ';'.join(by_pos.get(pos, ['H']))
    rows.append(entry)

subs_df = pd.DataFrame(rows)
display(subs_df)

,entry_idx,smiles,ligand_class,pos2,pos3,pos4,pos5
0,0,[H]C1=C(C(F)(F)F)C([H])=C([H])C(=C2N[C@@]([H])...,pyox,H,FC(F)F,H,H
1,1,[H]C1=C(OC([H])([H])c2c([H])c([H])c([H])c([H])...,pyox,H,OCc1ccccc1,H,H
2,2,[H]C1=C(C(F)(F)F)C([H])=C([H])C(=C2N[C@@]([H])...,pyox,H,FC(F)F,H,H
3,3,[H]C1=C([H])C(=C2N[C@@]([H])(C([H])([H])c3c([H...,pyox,C,H,H,H
4,4,[H]C1=C(C(F)(F)F)C([H])=C([H])C(=C2N[C@@]3([H]...,pyox,H,FC(F)F,H,H
5,5,[H]C1=C(C(F)(F)F)C([H])=C([H])C(=C2N[C@@]([H])...,pyox,H,FC(F)F,H,H
6,6,[H]C1=C(C(F)(F)F)C([H])=C([H])C(=C2N[C@@]([H])...,pyox,H,FC(F)F,H,H
7,7,[H]C1=C(OC([H])([H])c2c([H])c([H])c([H])c([H])...,pyox,H,OCc1ccccc1,H,H
8,8,[H]C1=C(C(F)(F)F)C([H])=C([H])C(=C2N[C@@]([H])...,pyox,H,FC(F)F,H,H
9,9,[H]C1=C(F)NC(=C2N[C@@]3([H])c4c([H])c([H])c([H...,pyox,F,H,H,H


In [ ]:
# Wide DataFrame including both pyridine (pos2-5) and oxazoline (pos6-7) positions
ALL_POSITIONS = [2, 3, 4, 5, 6, 7]

rows = []
for idx, row in df.iterrows():
    pyr_subs = row['pyridine_subs']
    ox_subs = row['oxazoline_subs']
    combined = (pyr_subs or []) + (ox_subs or [])

    entry = {
        'entry_idx': idx,
        'smiles': row['smiles'],
        'ligand_class': row.get('ligand_class'),
    }
    by_pos = defaultdict(list)
    for pos, smi in combined:
        by_pos[pos].append(smi)
    for pos in ALL_POSITIONS:
        if pos in by_pos:
            entry[f'pos{pos}'] = ';'.join(by_pos[pos])
        elif pyr_subs is None and pos in (2, 3, 4, 5):
            entry[f'pos{pos}'] = None
        elif ox_subs is None and pos in (6, 7):
            entry[f'pos{pos}'] = None
        else:
            entry[f'pos{pos}'] = 'H'
    rows.append(entry)

full_subs_df = pd.DataFrame(rows)
display(full_subs_df)

In [ ]:
def filter_by_substituent(df, position, substituent):
    """
    Return the subset of df (subs_df or full_subs_df) where the given ring
    position bears the given substituent.

    position: ring position number (2-5 for pyridine, 6-7 for oxazoline,
              matching the posN columns in df)
    substituent: substituent SMILES fragment (e.g. 'F', 'FC(F)F') or 'H' for
                 an unsubstituted position. Matched after canonicalization, so
                 equivalent SMILES spellings (e.g. 'C(F)(F)F' vs 'FC(F)F') are
                 treated as the same substituent.
    """
    col = f'pos{position}'
    if col not in df.columns:
        raise ValueError(f'No column {col!r} in dataframe; valid positions are {[c[3:] for c in df.columns if c.startswith("pos")]}')

    def canon(smi):
        if smi == 'H':
            return 'H'
        mol = Chem.MolFromSmiles(smi)
        return Chem.MolToSmiles(mol) if mol is not None else smi

    target = canon(substituent)

    def matches(cell):
        if pd.isna(cell):
            return False
        return target in {canon(part) for part in cell.split(';')}

    return df[df[col].apply(matches)]


# Example: all entries with CF3 at position 3
display(filter_by_substituent(full_subs_df, 3, 'FC(F)F'))

# Example: all entries unsubstituted (H) at position 2
display(filter_by_substituent(full_subs_df, 2, 'H'))

In [ ]:
def export_filtered_entries(filtered_df, original_df=df, filename=None):
    """
    Match the entry_idx values in filtered_df (a subset of subs_df/full_subs_df
    produced by filter_by_substituent) back to the corresponding rows of the
    original spreadsheet, and write those rows to a new CSV.

    If filename is not provided, prompts for one interactively. A '.csv'
    extension is appended if missing.
    """
    matched = original_df.loc[filtered_df['entry_idx']]

    if filename is None:
        filename = input('Enter output CSV filename: ').strip()
    if not filename.lower().endswith('.csv'):
        filename += '.csv'

    matched.to_csv(filename, index=False)
    print(f'Wrote {len(matched)} entries to {filename}')
    return matched


# Example: export all entries with CF3 at position 3
cf3_pos3 = filter_by_substituent(full_subs_df, 3, 'FC(F)F')
export_filtered_entries(cf3_pos3)